# Attention Mechanisms
At this point, we know how to prepare the input text for training LLMs by splitting text into individual word and subwords.

## Modelling Long Sequences and Why That's Hard
Think about translating a german sentence into english - we can't just go word by word because languages have different structures...

Before Transformers we would use an RNN.

In an **encoder–decoder RNN**, the input text is fed into the encoder, which processes it sequentially. The encoder updates its hidden state (the internal values at the hidden layers) at each step, trying to capture the entire meaning of the input sentence in the final hidden state, as illustrated in figure 3.4. The decoder then takes this final hidden state to start generating the translated sentence, one word at a time. It also updates its hidden state at each step, which is supposed to carry the context necessary for the next-word prediction.

The big limitation of the RNN architecture is that the RNN can't directly access earlier hidden states from the encoder during the decoding phase.

Because of this quirk, RNNs don't work for long texts because they don't have access to previous words in the input.

### Bahdanau Attention
Bahdanau Attention modified the encoder–decoder RNN such that the decoder can selectively access different parts of the input sequence at each decoding step.

Interestingly, researchers eventually realized that we don't need to use any RNN architecture at all, instead developing an architecture reliant on *self-attention*.

> Note: The "self" in self-attention refers to the fact that the attention mechanism can compute attention weights by relating different positions within a single input sequence, rather than comparing attention across separate sequences (like seq2seq)

## Self Attention w/o Trainable Weights
In self-attention, our goal is to calculate context vectors $z^{(i)}$ for each element $x^{(i)}$ in the input sequence. A context vector can be interpreted as an enriched embedding vector.

In [1]:
import torch
inputs = torch.tensor(
    [[0.43, 0.15, 0.89], # Your     (x^1)
    [0.55, 0.87, 0.66], # journey 
    [0.57, 0.85, 0.64], # starts
    [0.22, 0.58, 0.33], # with
    [0.77, 0.25, 0.10], # one
    [0.05, 0.80, 0.55]] # step
)

In [2]:
query = inputs[1]
attn_scores_2 = torch.empty(inputs.shape[0]) #w2

In [3]:
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(query, x_i)
print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


Beyond viewing the dot product operation as a mathematical tool that combines two vectors to yield a scalar value, the dot product is a measure of similarity because it quantifies how closely two vectors are aligned: a higher dot product indicates a greater degree of alignment or similarity between the vectors. In the context of self-attention mechanisms, the dot product determines the extent to which each element in a sequence focuses on, or “attends to,” any other element: the higher the dot 
product, the higher the similarity and attention score between two elements.

We now want to normalize...

In [4]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
attn_weights_2_tmp

# softmax is better for this tho usually
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

# note however our implementation of softmax can be unstable for extreme values, so let's use pytorch's
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)
Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


Now that we have computed the normalized attention weights, we are ready for the final step, as shown in figure 3.10: calculating the context vector z(2) by multiplying the embedded input tokens, x(i), with the corresponding attention weights and then sum- ming the resulting vectors. Thus, context vector z(2) is the weighted sum of all input vec- tors, obtained by multiplying each input vector by its corresponding attention weight:

In [5]:
query = inputs[1]

context_vector_2 = torch.zeros(query.shape)

for i, x_i in enumerate(inputs):
    context_vector_2 += x_i * attn_weights_2[i]
context_vector_2

tensor([0.4419, 0.6515, 0.5683])

now to make a self-attention matrix:

In [6]:
attn_scores = torch.empty(6, 6)

for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(inputs[i], inputs[j])

attn_scores

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

but this is hella slow! poo. what we're actually doing with a double for loop is essentially just a dot product with a transpose of the vector:

In [7]:
attn_scores_vectorized = inputs @ inputs.T

attn_scores_vectorized

# yipper!!!

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [8]:
# now let's also apply softmax row-wise
attn_weights = torch.softmax(attn_scores_vectorized, dim = -1)

attn_weights

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [9]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [11]:
# fuck it practice
import torch
import tiktoken

# let's say we encode:

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    test_text = f.read()

tokenizer = tiktoken.get_encoding("gpt2")
encoded_text = tokenizer.encode(test_text)

d = 256
vocab_size = tokenizer.n_vocab
embedding_layer = torch.nn.Embedding(vocab_size, d)
embedded_text = embedding_layer(torch.tensor(encoded_text))

embedded_text.shape

self_attention_matrix = embedded_text @ embedded_text.T
weight_matrix = torch.softmax(self_attention_matrix, dim = -1)

weight_matrix

FileNotFoundError: [Errno 2] No such file or directory: 'the-verdict.txt'

## Self-attention with trainable weights

In [31]:
x_2 = inputs[1]
d_in = inputs.shape[-1]
d_out = 2

torch.manual_seed(123)

# making our weight matrices
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad = False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad = False)

query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print(query_2)

keys = inputs @ W_key
values = inputs @ W_value
print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

# attention score 22
key_2 = keys[1]
attn_score_22 = query_2.dot(key_2)
print(attn_score_22)

# attention score for all tokens (w.r.t token 2)
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

# now we want to softmax them!
d_k = keys.shape[-1]
attn_scores_2 = torch.softmax(attn_scores_2 / d_k ** 0.5, dim = -1)
print(attn_scores_2)

tensor([0.4306, 1.4551])
keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])
tensor(1.8524)
tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])
tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


**The rationale behind scaled-dot product attention**

The reason for the normalization by the embedding dimension size is to improve the
training performance by **avoiding small gradients**. For instance, when scaling up the
embedding dimension, **which is typically greater than 1,000 for GPT-like LLMs, large
dot products can result in very small gradients during backpropagation due to the
softmax function applied to them.** As dot products increase, the softmax function
behaves more like a step function, resulting in gradients nearing zero. These small
gradients can drastically slow down learning or cause training to stagnate.
The scaling by the square root of the embedding dimension is the reason why this
self-attention mechanism is also called scaled-dot product attention. 

In [30]:
# finally, to get the context vector
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3069, 0.8188])


In [ ]:
# now let's make a class
import torch.nn as nn
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        


_IncompleteInputError: incomplete input (1347308845.py, line 4)